# Super Model Evaluation Notebook

This super notebook aggregates the end-to-end pipeline for multivariate time series classification on the Chapman ECG dataset.
It covers:
- Data loading and dataset splitting.
- MultiROCKET feature extraction.
- Dimensionality Reduction via Autoencoder (Path A) and Structured Pooling (Path B).
- Model Training across three archtiectures:
  - **MLP**: A1 (features from Autoencoder), B1 (features from Pooling)
  - **FT-Transformer**: A2, B2
  - **ConvTran** (with AMP): A3 (Autoencoder), B3 (Pooling)
- Exploration of **Class Imbalance Handling**: evaluating all conditions with and without balanced class weights.
- A final comprehensive evaluation table.


In [1]:
import os, sys, gc, logging, copy
from pathlib import Path
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from tqdm import tqdm
from sklearn.utils.class_weight import compute_class_weight

# Basic setup & Paths
PROJECT_ROOT = Path.cwd().resolve().parents[0]
project_root_str = str(PROJECT_ROOT)
convtran_root_str = str(PROJECT_ROOT / 'ConvTran')

if project_root_str not in sys.path:
    sys.path.insert(0, project_root_str)
if convtran_root_str not in sys.path:
    sys.path.insert(0, convtran_root_str)

# Reload src module safely
for name in list(sys.modules.keys()):
    if name == "src" or name.startswith("src."):
        del sys.modules[name]

from src.utils import SEED, set_global_seed
from src.utils.paths import get_reduced_dir, ensure_dir, get_experiment_dir, get_experiment_output_dir
from src.data import CLASS_NAMES, load_labels
from src.data.splits import load_splits
from src.training.evaluation import compute_metrics
from src.models.mlp import MLPClassifier
from src.models.ft_transformer import FTTransformer
from Models.model import model_factory  # ConvTran
from src.data.smoteenn_resampling import apply_smoteenn_to_reduced_features
from src.training.class_balanced_focal import make_cb_focal_loss
from src.training.weighted_ce import make_weighted_ce_loss

logging.basicConfig(level=logging.INFO)
NUM_CLASSES = len(CLASS_NAMES)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")


Using device: cuda


## 1. Data Preparation
We load the train/val/test splits and their corresponding labels.


In [2]:
print("Loading targets and splits...")
idx_train, idx_val, idx_test = load_splits()
try:
    y_full = load_labels()
except FileNotFoundError:
    from src.data import load_raw_dataset
    _, y_full = load_raw_dataset()

y_idx = np.argmax(y_full, axis=1) if y_full.ndim == 2 else y_full
y_train = y_idx[idx_train]
y_test = y_idx[idx_test]
y_val = y_idx[idx_val] if len(idx_val) > 0 else None

print(f"Train samples: {len(y_train)}, Val samples: {len(y_val) if y_val is not None else 0}, Test samples: {len(y_test)}")


Loading targets and splits...
Train samples: 36120, Val samples: 2257, Test samples: 6773


## 2. Feature Loading
In this notebook, we load the pre-computed reduced features to save time. If they are not available, the appropriate pipelines (MultiRocket -> Autoencoder/Pooling) would be run.
- **Autoencoder Features**: Shape is typically `(N, 256)`
- **Pooled Features**: Shape is typically `(N, 176)`


In [3]:
def safe_load(path, required=False):
    if path.exists():
        return np.load(path).astype(np.float32)
    if required:
        raise FileNotFoundError(f"{path} not found. Please run preprocessing first.")
    return None

red_dir_ae = get_reduced_dir() / "autoencoder"
red_dir_pool = get_reduced_dir() / "pooled"

features = {
    'autoencoder': {
        'train': safe_load(red_dir_ae / "train.npy"),
        'val': safe_load(red_dir_ae / "val.npy"),
        'test': safe_load(red_dir_ae / "test.npy")
    },
    'pool': {
        'train': safe_load(red_dir_pool / "train.npy"),
        'val': safe_load(red_dir_pool / "val.npy"),
        'test': safe_load(red_dir_pool / "test.npy")
    }
}

if features['autoencoder']['train'] is not None:
    print(f"Autoencoder Input Dim: {features['autoencoder']['train'].shape[1]}")
if features['pool']['train'] is not None:
    print(f"Pooled Input Dim: {features['pool']['train'].shape[1]}")


Autoencoder Input Dim: 256
Pooled Input Dim: 2016


## 3. Generic Training Loop
We define a robust, adaptable training loop that handles MLP, FT-Transformer, and ConvTran using Early Stopping.


In [4]:
def train_and_evaluate(model_type, feature_type, imb_method):
    set_global_seed(SEED)
    
    # Configure names & features
    if model_type in ['A1', 'A2', 'A3']:
        f_key = 'autoencoder'
    else:
        f_key = 'pool'
        
    X_tr = features[f_key]['train']
    X_vl = features[f_key]['val']
    X_ts = features[f_key]['test']
    input_dim = X_tr.shape[1]

    cond_name = f"{model_type}_{f_key}_{imb_method}_super"
    print(f"\n{'='*50}")
    print(f"Training {cond_name}")
    
    exp_dir = ensure_dir(get_experiment_dir(cond_name))
    ckpt_dir = ensure_dir(get_experiment_output_dir(cond_name, checkpoints=True))
    ckpt_path = ckpt_dir / "model.pt"
    
    # Imbalance handling
    y_train_local = y_train.copy()
    if imb_method == 'SMOTEENN':
        X_tr, y_train_local = apply_smoteenn_to_reduced_features(X_tr, y_train_local)
        print(f'  -> Applied SMOTEENN. Synthesized training set shape: {X_tr.shape}')
    
    if imb_method == 'Focal':
        criterion = make_cb_focal_loss(y_train_local, NUM_CLASSES, device=device)
    elif imb_method == 'WeightedCE':
        criterion = make_weighted_ce_loss(y_train_local, NUM_CLASSES, device=device)
    else:
        criterion = torch.nn.CrossEntropyLoss()
    
    # Architecture
    if model_type in ['A1', 'B1']:  # MLP
        model = MLPClassifier(input_dim=input_dim, num_classes=NUM_CLASSES, hidden_dims=[1024, 512, 256, 128], dropout=0.2).to(device)
        lr, bs, eps = 1e-3, 256, 80
    elif model_type in ['A2', 'B2']:  # FT-Transformer
        model = FTTransformer(n_features=input_dim, num_classes=NUM_CLASSES, d_token=64, n_heads=4, n_layers=3, dropout=0.1).to(device)
        lr, bs, eps = 2e-3, 1024, 20
    elif model_type in ['A3', 'B3']:  # ConvTran
        data_shape = (X_tr.shape[0], 1, X_tr.shape[1])
        config = {'Net_Type': ['C-T'], 'emb_size': 16, 'dim_ff': 256, 'num_heads': 8, 'Fix_pos_encode': 'tAPE', 'Rel_pos_encode': 'eRPE', 'dropout': 0.1, 'Data_shape': data_shape, 'num_labels': NUM_CLASSES}
        model = model_factory(config).to(device)
        lr, bs, eps = 1e-4, 64, 30
    
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
    scaler = torch.cuda.amp.GradScaler() if model_type in ['A3', 'B3'] else None
    
    best_val_f1 = -1
    epochs_no_improve = 0
    patience = 5 if model_type in ['A3', 'B3'] else 10
    start_epoch = 0
    
    # Attempt to load checkpoint if it exists to resume
    if ckpt_path.exists():
        try:
            print(f"Found existing checkpoint at {ckpt_path}. Loading...")
            checkpoint = torch.load(ckpt_path, map_location=device)
            model.load_state_dict(checkpoint['model_state'])
            opt.load_state_dict(checkpoint['optimizer_state'])
            if scaler and 'scaler_state' in checkpoint:
                scaler.load_state_dict(checkpoint['scaler_state'])
            best_val_f1 = checkpoint.get('best_val_f1', -1)
            epochs_no_improve = checkpoint.get('epochs_no_improve', 0)
            start_epoch = checkpoint.get('epoch', 0) + 1
            print(f"Resumed from epoch {start_epoch} with Best Val F1: {best_val_f1:.4f}")
        except Exception as e:
            print(f"Failed to load checkpoint: {e}. Starting fresh.")
    
    best_wts = copy.deepcopy(model.state_dict())
    
    n_train = len(X_tr)
    
    try:
        for ep in range(start_epoch, eps):
            model.train()
            perm = np.random.permutation(n_train)
            total_loss = 0
            nb = 0
            pbar = tqdm(range(0, n_train, bs), desc=f"Epoch {ep}", leave=False)
            for start in pbar:
                idx = perm[start:start+bs]
                bx = torch.from_numpy(X_tr[idx]).float().to(device)
                by = torch.from_numpy(y_train_local[idx]).long().to(device)
                
                if model_type in ['A3', 'B3']: bx = bx.unsqueeze(1)  # ConvTran expects 3D
                
                opt.zero_grad()
                if scaler:
                    with torch.cuda.amp.autocast():
                        logits = model(bx)
                        loss = criterion(logits, by)
                    scaler.scale(loss).backward()
                    scaler.step(opt)
                    scaler.update()
                else:
                    logits = model(bx)
                    loss = criterion(logits, by)
                    loss.backward()
                    opt.step()
                
                total_loss += loss.item()
                nb += 1
            
            # Simple Val loop step
            if bool(X_vl is not None and len(X_vl)): 
                model.eval()
                with torch.no_grad():
                    vl_x = torch.from_numpy(X_vl).float()
                    if model_type in ['A3', 'B3']: vl_x = vl_x.unsqueeze(1)
                    vl_logits = []
                    for idx_v in range(0, len(vl_x), bs*2):
                        vl_logits.append(model(vl_x[idx_v:idx_v+bs*2].to(device)).cpu())
                    vl_logits = torch.cat(vl_logits)
                    val_preds = vl_logits.argmax(dim=1).numpy()
                    val_f1 = compute_metrics(y_val, val_preds, task='multiclass', labels=list(range(NUM_CLASSES)))['weighted_f1']
                
                if val_f1 > best_val_f1:
                    best_val_f1 = val_f1
                    best_wts = copy.deepcopy(model.state_dict())
                    epochs_no_improve = 0
                    
                    # Save checkpoint
                    torch.save({
                        'epoch': ep,
                        'model_state': model.state_dict(),
                        'optimizer_state': opt.state_dict(),
                        'scaler_state': scaler.state_dict() if scaler else None,
                        'best_val_f1': best_val_f1,
                        'epochs_no_improve': epochs_no_improve
                    }, ckpt_path)
                    print(f"  [Epoch {ep}] New best Val F1: {best_val_f1:.4f} - Checkpoint saved.")
                else:
                    epochs_no_improve += 1
                    
                    # Periodically save resume checkpoints even if not best
                    torch.save({
                        'epoch': ep,
                        'model_state': model.state_dict(),
                        'optimizer_state': opt.state_dict(),
                        'scaler_state': scaler.state_dict() if scaler else None,
                        'best_val_f1': best_val_f1,
                        'epochs_no_improve': epochs_no_improve
                    }, ckpt_dir / "model_latest.pt")
                    
                    if epochs_no_improve >= patience:
                        print(f"  Early stopping triggered after {patience} epochs without improvement.")
                        break
        print("Training loop finished gracefully.")
    except KeyboardInterrupt:
        print("\nTraining interrupted by user. Gracefully handling...")
    except Exception as e:
        print(f"\nCrash encountered: {e}. Gracefully handling...")
    finally:
        # Guarantee best weights are reloaded
        try:
            model.load_state_dict(best_wts)
        except Exception as rollback_err:
            print(f"Could not load best in-memory weights: {rollback_err}")
        
        # Test Eval
        model.eval()
        with torch.no_grad():
            ts_x = torch.from_numpy(X_ts).float()
            if model_type in ['A3', 'B3']: ts_x = ts_x.unsqueeze(1)
            ts_logits = []
            for idx_t in range(0, len(ts_x), bs*2):
                ts_logits.append(model(ts_x[idx_t:idx_t+bs*2].to(device)).cpu())
            ts_logits = torch.cat(ts_logits)
            test_preds = ts_logits.argmax(dim=1).numpy()
            from sklearn.metrics import precision_recall_fscore_support
            p, r, f, _ = precision_recall_fscore_support(y_test, test_preds, labels=list(range(NUM_CLASSES)), average='weighted', zero_division=0)
        
        print(f"Done. Test Precision: {p:.4f} | Recall: {r:.4f} | F1: {f:.4f}")
        
        # Guarantee GPU freeing when execution stops or crashes
        if 'logits' in locals(): del logits
        if 'val_preds' in locals(): del val_preds
        if 'ts_logits' in locals(): del ts_logits
        gc.collect()
        torch.cuda.empty_cache()
        
        return {'P': p, 'R': r, 'F1': f}


## 4. Run Experiment Suite
Here we trigger the execution for all configurations.


In [ ]:
results = []
configs = ['A1', 'A2', 'A3', 'B1', 'B2', 'B3']

for c in configs:
    for imb_m in ['None', 'WeightedCE', 'Focal', 'SMOTEENN']:
        if features['autoencoder']['train'] is None and c in ['A1','A2','A3']:
            print(f"Skipping {c} due to missing AE features")
            continue
        if features['pool']['train'] is None and c in ['B1','B2','B3']:
            print(f"Skipping {c} due to missing Pool features")
            continue
        
        mets = train_and_evaluate(c, 'autoencoder' if c in ['A1','A2','A3'] else 'pool', imb_m)
        results.append({
            'Model Config': c,
            'Architecture': 'MLP' if '1' in c else ('FT-Transformer' if '2' in c else 'ConvTran'),
            'Features': 'Autoencoder' if c in ['A1','A2','A3'] else 'Pooling',
            'Class Imbalance Method': imb_m,
            'Test Weighted Precision': mets['P'],
            'Test Weighted Recall': mets['R'],
            'Test Weighted F1': mets['F1']
        })



INFO:src.utils.seed:Global seed set to 0



Training A1_autoencoder_None_super
Found existing checkpoint at C:\Users\hmanasi1\Documents\deeplearning\Deep-Learning-Project\experiments\A1_autoencoder_None_super\checkpoints\model.pt. Loading...
Resumed from epoch 15 with Best Val F1: 0.9186


  Early stopping triggered after 10 epochs without improvement.
Training loop finished gracefully.
Done. Test Precision: 0.9107 | Recall: 0.9108 | F1: 0.9107


INFO:src.utils.seed:Global seed set to 0



Training A1_autoencoder_WeightedCE_super
Found existing checkpoint at C:\Users\hmanasi1\Documents\deeplearning\Deep-Learning-Project\experiments\A1_autoencoder_WeightedCE_super\checkpoints\model.pt. Loading...
Resumed from epoch 47 with Best Val F1: 0.9118


...


Training interrupted by user. Gracefully handling...
Done. Test Precision: 0.1944 | Recall: 0.4409 | F1: 0.2698


INFO:src.utils.seed:Global seed set to 0



Training B2_pool_Focal_super


  [Epoch 0] New best Val F1: 0.1936 - Checkpoint saved.


Epoch 1:   0%|                                                                                  | 0/36 [00:00<?, ?it/s]

In [5]:
results = []
configs = ['B3']

for c in configs:
    for imb_m in ['None', 'WeightedCE']:
        if features['autoencoder']['train'] is None and c in ['A1','A2','A3']:
            print(f"Skipping {c} due to missing AE features")
            continue
        if features['pool']['train'] is None and c in ['B1','B2','B3']:
            print(f"Skipping {c} due to missing Pool features")
            continue
        
        mets = train_and_evaluate(c, 'autoencoder' if c in ['A1','A2','A3'] else 'pool', imb_m)
        results.append({
            'Model Config': c,
            'Architecture': 'MLP' if '1' in c else ('FT-Transformer' if '2' in c else 'ConvTran'),
            'Features': 'Autoencoder' if c in ['A1','A2','A3'] else 'Pooling',
            'Class Imbalance Method': imb_m,
            'Test Weighted Precision': mets['P'],
            'Test Weighted Recall': mets['R'],
            'Test Weighted F1': mets['F1']
        })

INFO:src.utils.seed:Global seed set to 0
C:\Users\hmanasi1\AppData\Local\anaconda3\envs\dl_project\lib\site-packages\torch\functional.py:505: UserWarning: torch.meshgrid: in an upcoming release, it will be required to pass the indexing argument. (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\pytorch\aten\src\ATen\native\TensorShape.cpp:4383.)
  return _VF.meshgrid(tensors, **kwargs)  # type: ignore[attr-defined]



Training B3_pool_None_super


C:\Users\hmanasi1\AppData\Local\Temp\ipykernel_30428\807372570.py:50: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler() if model_type in ['A3', 'B3'] else None


Found existing checkpoint at C:\Users\hmanasi1\Documents\deeplearning\Deep-Learning-Project\experiments\B3_pool_None_super\checkpoints\model.pt. Loading...
Resumed from epoch 3 with Best Val F1: 0.5985


Epoch 3:   0%|                                                                                 | 0/565 [00:00<?, ?it/s]C:\Users\hmanasi1\AppData\Local\Temp\ipykernel_30428\807372570.py:93: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast():
C:\Users\hmanasi1\AppData\Local\anaconda3\envs\dl_project\lib\site-packages\torch\nn\modules\conv.py:548: UserWarning: Using padding='same' with even kernel lengths and odd dilation may require a zero-padded copy of the input be created (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\pytorch\aten\src\ATen\native\Convolution.cpp:1026.)
  return F.conv2d(
                                                                                                                       

...

  [Epoch 1] New best Val F1: 0.5968 - Checkpoint saved.


Epoch 2:   0%|                                                                                 | 0/565 [00:00<?, ?it/s]C:\Users\hmanasi1\AppData\Local\Temp\ipykernel_30428\807372570.py:93: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast():
                                                                                                                       

  [Epoch 2] New best Val F1: 0.6005 - Checkpoint saved.


Epoch 3:   0%|                                                                                 | 0/565 [00:00<?, ?it/s]C:\Users\hmanasi1\AppData\Local\Temp\ipykernel_30428\807372570.py:93: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast():
                                                                                                                       


Training interrupted by user. Gracefully handling...
Done. Test Precision: 0.6346 | Recall: 0.5678 | F1: 0.5885


## 5. Final Results Table
Consolidates all metrics into one summary view.


In [6]:
df_results = pd.DataFrame(results)
from IPython.display import display
display(df_results.sort_values(by=['Test Weighted F1'], ascending=False))
df_results.to_csv('super_model_evaluation_results.csv', index=False)


,Model Config,Architecture,Features,Class Imbalance Method,Test Weighted Precision,Test Weighted Recall,Test Weighted F1
1,B3,ConvTran,Pooling,WeightedCE,0.634628,0.567843,0.588544
0,B3,ConvTran,Pooling,None,0.571880,0.615237,0.584330


In [7]:
import os
import torch
import numpy as np
import pandas as pd
from tqdm import tqdm
from pathlib import Path
from sklearn.metrics import precision_recall_fscore_support

# 1. Setup paths and find super folders
PROJECT_ROOT = Path.cwd().resolve().parents[0]
exp_base = PROJECT_ROOT / "experiments"
super_folders = [f for f in exp_base.iterdir() if f.is_dir() and f.name.endswith("_super")]

results = []

print(f"Found {len(super_folders)} super experiment folders. Starting reconstruction...")

for folder in tqdm(super_folders):
    cond_name = folder.name
    # Parse metadata from folder name (e.g., A1_autoencoder_Focal_super)
    parts = cond_name.split('_')
    if len(parts) < 4: continue
    
    model_type = parts[0]
    f_key = parts[1]
    imb_method = parts[2]
    
    ckpt_path = folder / "checkpoints" / "model.pt"
    if not ckpt_path.exists():
        continue
        
    # 2. Load the features for this specific architecture
    X_ts = features[f_key]['test']
    input_dim = X_ts.shape[1]
    
    # 3. Re-initialize the correct Architecture
    if model_type in ['A1', 'B1']:  # MLP
        model = MLPClassifier(input_dim=input_dim, num_classes=NUM_CLASSES, hidden_dims=[1024, 512, 256, 128], dropout=0.2).to(device)
    elif model_type in ['A2', 'B2']:  # FT-Transformer
        model = FTTransformer(n_features=input_dim, num_classes=NUM_CLASSES, d_token=64, n_heads=4, n_layers=3, dropout=0.1).to(device)
    elif model_type in ['A3', 'B3']:  # ConvTran
        data_shape = (1, 1, input_dim) # dummy shape for init
        cfg = {'Net_Type': ['C-T'], 'emb_size': 16, 'dim_ff': 256, 'num_heads': 8, 'Fix_pos_encode': 'tAPE', 'Rel_pos_encode': 'eRPE', 'dropout': 0.1, 'Data_shape': data_shape, 'num_labels': NUM_CLASSES}
        model = model_factory(cfg).to(device)
    
    # 4. Load weights
    try:
        checkpoint = torch.load(ckpt_path, map_location=device)
        model.load_state_dict(checkpoint['model_state'])
        model.eval()
        
        # 5. Evaluate on Test Set
        with torch.no_grad():
            ts_x = torch.from_numpy(X_ts).float()
            if model_type in ['A3', 'B3']: ts_x = ts_x.unsqueeze(1)
            
            # Batch inference to avoid OOM
            bs = 256
            ts_logits = []
            for idx_t in range(0, len(ts_x), bs):
                batch = ts_x[idx_t : idx_t + bs].to(device)
                ts_logits.append(model(batch).cpu())
            
            ts_logits = torch.cat(ts_logits)
            test_preds = ts_logits.argmax(dim=1).numpy()
            
            p, r, f, _ = precision_recall_fscore_support(
                y_test, test_preds, labels=list(range(NUM_CLASSES)), 
                average='weighted', zero_division=0
            )
            
        results.append({
            'Model Config': model_type,
            'Architecture': 'MLP' if '1' in model_type else ('FT-Transformer' if '2' in model_type else 'ConvTran'),
            'Features': f_key.capitalize(),
            'Class Imbalance Method': imb_method,
            'Test Weighted Precision': p,
            'Test Weighted Recall': r,
            'Test Weighted F1': f
        })
        
    except Exception as e:
        print(f"Error loading {cond_name}: {e}")

# 6. Display Reconstructed Table
df_reconstructed = pd.DataFrame(results)
display(df_reconstructed.sort_values(by=['Test Weighted F1'], ascending=False))

# Optional: Save it so you don't have to run this again
df_reconstructed.to_csv('super_model_evaluation_results.csv', index=False)


Found 25 super experiment folders. Starting reconstruction...


100%|██████████████████████████████████████████████████████████████████████████████████| 25/25 [12:55<00:00, 31.03s/it]


,Model Config,Architecture,Features,Class Imbalance Method,Test Weighted Precision,Test Weighted Recall,Test Weighted F1
1,A1,MLP,Autoencoder,None,0.910714,0.910822,0.910714
3,A1,MLP,Autoencoder,unweighted,0.910714,0.910822,0.910714
4,A1,MLP,Autoencoder,WeightedCE,0.906021,0.898420,0.901011
5,A1,MLP,Autoencoder,weighted,0.907005,0.897977,0.900803
0,A1,MLP,Autoencoder,Focal,0.904653,0.889119,0.893699
2,A1,MLP,Autoencoder,SMOTEENN,0.887065,0.853536,0.861273
11,A3,ConvTran,Autoencoder,None,0.839479,0.861214,0.849023
17,B1,MLP,Pool,WeightedCE,0.855999,0.824007,0.830328
10,A3,ConvTran,Autoencoder,Focal,0.832888,0.792116,0.806116
13,A3,ConvTran,Autoencoder,WeightedCE,0.831086,0.764949,0.784791
